In [1]:
import torch
import torch.nn as nn
import math

In [2]:
device = "mps" if torch.backends.mps.is_available() else "cpu"

In [3]:
text = open("data/clean.txt", encoding="utf-8").read()

chars = sorted(set(text)) # separate each char
stoi = {ch: i for i, ch in enumerate(chars)} # 'a': 0, char to int (mapping)
itos = {i: ch for ch, i in stoi.items()} # 0 : 'a' , int to char (reverse mapping)
vocab_size = len(chars) # char size

data = torch.tensor([stoi[ch] for ch in text], dtype=torch.long) # to tensor

In [4]:
print(vocab_size)

76


In [ ]:
def batchify(data, batch_size):
    n = data.size(0) // batch_size  # n char per batch = char / batch_size
    data = data[: n * batch_size] # cutting
    return data.view(batch_size, n) # to list form (batch_size row with n elements per row)

def get_batches(source, seq_len):
    for i in range(0, source.size(1) - 1, seq_len): # From 0 to n with seq_len steps
        # print(i)
        length = min(seq_len, source.size(1) - 1 - i) # get length (noemallu is seq_size but sometime last round maybe smaller than seq_len)
        x = source[:, i : i + length] # from start to stop
        y = source[:, i + 1 : i + 1 + length] # next element of x sequence
        yield x, y

In [6]:
# Train Test Split
split = int(len(data) * 0.9)
train_data = batchify(data[:split], batch_size=32)
val_data = batchify(data[split:], batch_size=32)

In [7]:
class LSTMLanguageModel(nn.Module):
    def __init__(self, vocab_size, embed_dim=256, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.drop = nn.Dropout(dropout)
        self.lstm1 = nn.LSTM(embed_dim, 512, batch_first=True)
        self.lstm2 = nn.LSTM(512, embed_dim, batch_first=True)
        self.fc = nn.Linear(embed_dim, vocab_size)
        self.fc.weight = self.embedding.weight

    def forward(self, x, hidden=None):
        if hidden is None:
            hidden = [None, None]

        h = self.drop(self.embedding(x))
        h, h1 = self.lstm1(h, hidden[0])
        h = self.drop(h)
        h, h2 = self.lstm2(h, hidden[1])
        h = self.drop(h)

        return self.fc(h), [h1, h2]


def detach_hidden(hidden):
    return [(h.detach(), c.detach()) for h, c in hidden]

@torch.no_grad() #not train me pleaseeeee
def evaluate(model, data, seq_len=128):
    model.eval()
    total, count, hidden = 0.0, 0, None
    for x, y in get_batches(data, seq_len):
        x, y = x.to(device), y.to(device)
        logits, hidden = model(x, hidden)
        loss = loss_fn(logits.reshape(-1, vocab_size), y.reshape(-1))
        total += loss.item() * y.numel()
        count += y.numel()
    return total / count

In [8]:
model = LSTMLanguageModel(vocab_size).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3)
loss_fn = nn.CrossEntropyLoss()
best_val = float("inf")

# Parameter Setting
seq_len = 128
epochs = 40

for epoch in range(epochs):
    model.train()
    hidden, total, steps = None, 0.0, 0
    for x, y in get_batches(train_data, seq_len):
        x, y = x.to(device), y.to(device)
        if hidden is not None:
            hidden = detach_hidden(hidden)

        logits, hidden = model(x, hidden)
        loss = loss_fn(logits.reshape(-1, vocab_size), y.reshape(-1))

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)
        optimizer.step()

        total += loss.item()
        steps += 1

    train_loss = total / steps
    val_loss = evaluate(model, val_data, seq_len)
    print(f"epoch {epoch:2d}  train {train_loss:.3f}  val {val_loss:.3f} ppl {math.exp(val_loss):.2f}")

    # Best pleaseeeee
    if val_loss < best_val:
        best_val = val_loss
        torch.save({"model_state": model.state_dict(), "vocab_size": vocab_size,
                    "stoi": stoi, "itos": itos}, "lstm_lm_best.pt")

0
128
256
384
512
640
768
896
1024
1152
1280
1408
1536
1664
1792
1920
2048
2176
2304
2432
2560
2688
2816
2944
3072
3200
3328
3456
3584
3712
3840
3968
4096
4224
4352
4480
4608
4736
4864
4992
5120
5248
5376
5504
5632
5760
5888
6016
6144
6272
6400
6528
6656
6784
6912
7040
7168
7296
7424
7552
7680
7808
7936
8064
8192
8320
8448
8576
8704
8832
8960
9088
9216
9344
9472
9600
9728
9856
9984
10112
10240
10368
10496
10624
10752
10880
11008
11136
11264
11392
11520
11648
11776
11904
12032
12160
12288
12416
12544
12672
12800
12928
13056
13184
13312
13440
13568
13696
13824
13952
14080
14208
14336
14464
14592
14720
14848
14976
15104
15232
15360
15488
15616
15744
15872
16000
16128
16256
16384
16512
16640
16768
16896
17024
17152
17280
17408
17536
17664
17792
17920
18048
18176
18304
18432
18560
18688
18816
18944
19072
19200
19328
19456
19584
19712
19840
19968
20096
20224
20352
20480
20608
20736
20864
20992
21120
21248
21376
0
128
256
384
512
640
768
896
1024
1152
1280
1408
1536
1664
1792
1920
2048
2176
2

In [9]:
def text_to_ids(text):
    ids = [stoi[ch] for ch in text if ch in stoi]
    if not ids:
        raise ValueError("Char not in train data")
    return ids

def to_tensor(ids):
    return torch.tensor([ids], device=device)

def pick_next_char(scores, temperature, top_k):
    scores = scores / temperature

    if top_k:
        cutoff = torch.topk(scores, top_k).values[-1]
        scores = scores.masked_fill(scores < cutoff, -float("inf"))

    probs = torch.softmax(scores, dim=-1)
    return torch.multinomial(probs, 1).item()

@torch.no_grad()
def generate(model, prompt, length=500, temperature=0.8, top_k=20):
    model.eval()

    prompt_ids = text_to_ids(prompt)

    scores, memory = model(to_tensor(prompt_ids))
    last_scores = scores[0, -1]

    result = prompt
    for _ in range(length):
        next_id = pick_next_char(last_scores, temperature, top_k)
        result += itos[next_id]

        scores, memory = model(to_tensor([next_id]), memory)
        last_scores = scores[0, -1]

    return result

In [10]:
# load check point and to it.
ckpt = torch.load("lstm_lm_best.pt", map_location=device)

stoi, itos = ckpt["stoi"], ckpt["itos"]
model = LSTMLanguageModel(ckpt["vocab_size"]).to(device)
model.load_state_dict(ckpt["model_state"])

print(generate(model, "My father was a man of", length=400, temperature=0.8))

My father was a man of fear of feelings have a gild money as to the priate of the open table and the victim of every strange convention, the discourse of the party of the country was a note of maintaining their advantages. When her promise. He landed to see the village to every letter that we had a delegation, and seemed to such a present that the child, and I can all manager in one movement in the most of the characte
